In [ ]:
%pip install -q "ultralytics==8.4.174"

In [ ]:
from pathlib import Path
import subprocess
import torch
import ultralytics
from zipfile import ZipFile
from tqdm.auto import tqdm
from ultralytics import YOLO
import os
import json
import time

In [ ]:
ROOT = Path("/teamspace/studios/this_studio")
ZIP_DIR = ROOT / "detection_zips"
DATA = ROOT / "detection_prepared"

REMOTE = "gdrive:WasteScope/prepared/detection"
ZIP_DIR.mkdir(parents=True, exist_ok=True)

MOUNT = ROOT / "drive_runs"
DRIVE_RUNS = "gdrive:WasteScope/runs"
MOUNT.mkdir(parents=True, exist_ok=True)

for split in ("train", "val", "test"):
    print(f"Downloading {split}.zip", flush=True)
    subprocess.run(
        [
            "rclone", "copyto",
            f"{REMOTE}/{split}.zip",
            str(ZIP_DIR / f"{split}.zip"),
            "--progress",
            "--stats", "10s",
            "--buffer-size", "16M",
            "--multi-thread-streams", "4",
            "--retries", "5"
        ],
        check=True
    )

In [ ]:
if not os.path.ismount(MOUNT):
    subprocess.run(
        ["rclone", "mkdir", DRIVE_RUNS], check=True
    )
    subprocess.run(
        [
            "rclone", "mount", DRIVE_RUNS, str(MOUNT),
            "--vfs-cache-mode", "writes",
            "--vfs-write-back", "1s",
            "--cache-dir", str(ROOT / "rclone_runs_cache"),
            "--rc",
            "--rc-addr", "127.0.0.1:5578",
            "--daemon",
        ],
        check=True
    )

PROJECT = MOUNT / "detect"

In [ ]:
gpu = torch.cuda.get_device_properties(0)

print("Dataset:", DATA)
print("GPU:", gpu.name)
print(f"VRAM: {gpu.total_memory / 1024**3:.1f} GiB")
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Ultralytics:", ultralytics.__version__)

In [ ]:
DATA.mkdir(parents=True, exist_ok=True)
expected = {
    "train": 25228,
    "val": 6643,
    "test": 4607
}

for split, count in expected.items():
    with ZipFile(ZIP_DIR / f"{split}.zip") as archive:
        for member in tqdm(
            archive.infolist(), desc=f"Extracting {split}"
        ):
            archive.extract(member, DATA)
    
    images = {
        p.stem for p in (DATA / split / "images").glob("*.png")
    }
    labels = {
        p.stem for p in (DATA / split / "labels").glob("*.txt")
    }
    
    assert images == labels, f"{split}: image/label mismatch"
    assert len(images) == count, f"{split}: unexpected count"
    print(f"{split}: {len(images):,} image-label pairs — OK")

print("\nTraining-ready dataset:", DATA)

### D0 training

In [ ]:
model = YOLO("yolo26n.pt")

metrics = model.train(
    data=str(DATA / "data.yaml"),
    epochs=100,
    patience=20,
    imgsz=640,
    batch=64,
    device=0,
    workers=8,
    optimizer="auto",
    seed=42,
    deterministic=True,
    amp=True,
    cache=False,
    val=True,
    plots=True,
    save=True,
    project=str(PROJECT),
    name="D0_yolo26n_640",
    exist_ok=False
)

In [ ]:
RUN = Path(model.trainer.save_dir)

print("D0 validation results")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print("Best weights:", RUN / "weights/best.pt")
print("Last weights:", RUN / "weights/last.pt")
print("Training log:", RUN / "results.csv")

In [ ]:
while True:
    status = json.loads(subprocess.check_output(
        [
            "rclone", "rc", "vfs/stats",
            "--url", "http://127.0.0.1:5578",
        ],
        text=True
    ))["diskCache"]

    if status["outOfSpace"] or status["erroredFiles"]:
        raise RuntimeError(f"Drive upload problem: {status}")

    pending = status["uploadsQueued"] + status["uploadsInProgress"]

    if pending == 0:
        print("Drive upload complete.")
        break

    print(f"Waiting for {pending} uploads...", flush=True)
    time.sleep(5)